# Getting Started: Gemma 4 Developer Agent

## 1. Environment Configuration and Package Installation

In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')

## 2. Competition Dataset and Sample Submission

In [ ]:
import base64
import io
import re
import zipfile
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Our submission declares gemma-4-31b-it-qat-w4a16-ct in agent.yaml; no rewrite needed.
MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'

# EXP-002 shim check: unpack OUR submission (dist/submission.zip, sha256
# 33bebf151b2b4dfbce36c6933cfd9a5b10cf50229cc31d8a4513d8106f60f817) instead of sample_submission.
AGENT_DIR = WORKING_DIR / 'our_submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
AGENT_DIR.mkdir(parents=True)
SUBMISSION_B64 = "UEsDBBQAAAAIAAAAIVwJEt0MXAIAAAgEAAAKAAAAYWdlbnQueWFtbF1TTY/USAy9t9T/4cFceqRJtwYGDs0JaXuBw66gZ4XEKXKnnMSa+qLsTOhftP9jfxlKwiwNpypbLvu951dXOKZkoI6joU0FOpyCqEqKeLzFJpcUslUp+vMNVGLneSm+QUwgR9m4zHd9EO/1erteXeG+p8xok/dpVFjPeP/2+Pfh/r4+Ht7+8dcBQUpJBRQd2jQUNCmEIYqdL+YrNtlT3AaH//59ub3DpxdL98+H44c/v2w+vbjew0nbwnpRtDJDI4lqcGS0KzTulEL2XP/supvRb88UPFJseEY3lSsbROHSGH0ix+6XWbfXe1hKHpECK07s04gwqIFyZiqQOHdqUsjiuTIJjH9S8kfuRK2csekKZ5B7qC4kHntmP7Ga2u6hI9dNclzWq5Ac+z06DoGqu+rl7akSq76SVeMd3b6uGgPwqxYLVwQ6QzN7DzGMYj0IuaRHcVyQC7fy7c2PkxVUGGpFcmaH03km0VOJrLpeOdamSDZJcY8ja/KPrEiR8U7s/XCCqA78xP3j2foUUTgnFUvlDDLsxlQeNFPDT1iCRAnkb/DIRVphh0zW9Nv1alpdGZpl3DOJjR8cT9hDNt39L842uPWq48iFbEpF42jT2Up38W5J6GIBid288/Vq2qLu1yugQhliPVmPovuRYHL15KQlZCd2EY5FjC/ijq1WIxt0iefFWj3TmTJXeFco97NxFqUdN57KpPRgcBySsXuSb+GJzfPps0FazIZpSbw+v36DwyvQyZPx/KECUttuf8KYhKkjS9efUnmCw1SavlYJ4qnMJb890OHUTQjXq+9QSwMEFAAAAAgAAAAhXN159oETAgAAkAMAABUAAABjb25maWdzL3NhbXBsaW5nLnlhbWx9ks9u2zAMxu8B/A4fkEuCGc4/N2td9JBhHjas7bB2GLCTodi0I0SWNFFO6ifqe+zJBjlDMLTYdCFEUfz4IzlGQ5qc8FSURnvSPthaNqiNAx3I9RANaY+JVUInbYVfz2myAnvhvNQNDkJ1xNMkGo2xUcocqUItSVUMSw4fNw/3+eNj8ZBv3t/laKVzxmXw1Nog2zmK4Y0t7MnsY7TiqTCdt50vvNmT5hjWEZMuqbCkhfJ9HMRqRz870mV/9oJDCj65iWM4Yms0U9HKlgrfW4rBRFUMv5N6L3XzhzaJRn9VlGGeLHE+Y3BpHDlUVItOeUjGIplfI0+xmb3jIfow+KLRwBISXF2cLvsM6zQavaLKkM6v1kP6PZFlWGda6/EGpzAcSSl0uiIHvyOslvHb9SWGKT15HIVS18gXC3gnibGcp5ez9SJNo9ELtiwaAVKXqquo8DvTNTvPGbzrKIjXQjHh5twRmLrGhMPIBYO02Krw7fR4M0RPr5EvZ/kKYquEp/CVg8hZeNtVDfkMy4sAiDG+5w+fPvyYfF1NM5SmbTstfQ82nSsJjkQVhFn0PKDuhNPEjMoQ4/7Lt7CLR+Gqs9C/zhgnZXiDw+3tHSZGq/4lBIyembqexmDzetvCdEMRjoRCKWzyf8HPRBbSQ+j+KPos1N6qULysIRttXNi2rdQDoawhznSdrULzSNfGlcSQPolGvwFQSwMEFAAAAAgAAAAhXPryLAxGAgAAKgQAABAAAABldmFsX2NvbmZpZy55YW1sfZNNbhQxEIX3I80dnpQFMyLp+SEBMYEFC5CQEAk/ArHqVNo13da07ZarzKRPxD04GbInRAFBetWyy/W991w+wiXHEyXZ4TqZllUwG3rylTP4+eO0Op1X+NwxpAmRIyKTEfANNdqP0I6FsQ0pYsejIHnDEVf8nfpEaoPfXJ1PJ0cgOCtifZvL4Ji84P0FeuusHoO8yZ3QUfQsAsNbSr1i9nQJZ/0ii5tjH1JvcN2HfSlerdEhJn+rusqYL68/vn3zbfZhPd+gCX5roytAT47lDvOHPHgWzcKoJetFYUhpEWm/EHJDz7Wk66I9+EU+V5e+bTWS6wvza4iiaEgYA0fk6DZYrZfIqgU3mJ1WZ3iMVVXMQFjTMMdLrPJKhxd3fy0pY6Yh9LLIohwp17fuhnFeTSf3hE8ngKObWq3j2lmflGWDjAKOcpIGDQ0Ivlimlr2iD2HAjGT0jQ3IJ0PS+TnYWJVcaqzsQJEhavsebSTD5o4UQl831PeZkzGF1LLWoqTpkG9JS+uBtOlKp21kLjsmwAdFE5LXu5Ypetng2TIv3OqphZvgjWzwZLk8MO5fa864Cc4lb3WEhBQbLkOZ71A7K3j17tNF9i7F+WVHwifr4iWXDKOyFAH/+47yVB1DAvaMHfPwz9m0XpTJIGzLlT8SrM6WkMNbKXGfiDX8MCgbKVN5sA4reURmfyVxjMiOrC8O8zvN+/OiMEfUBFHJ2XbWt+cPA7OTIQY3KPYxR3SIA9bj6reI58sraMB1SN7AKsiPexqr6eQXUEsDBBQAAAAIAAAAIVyW5q1Dbg8AAJcmAAAUAAAAcHJvbXB0cy9zd2VfY29kZXIubWTdWm1z20aS/s5f0bX7QWSWhGXHye1RpatyYnmj2o3lspzb9V1dUUOgQcxqMAPPDEjx6n781dMzIEFLqShfl59AzFtPv/fT+Ox6Up4p7HhVuor9nBQFttp5+rCPjbPEdqMts6daP2i7oZv3V6RD6Jm0pdgwee5c0NH5PalIL3bO34dOlVzQT7qq2FLkEAPVzlNsdMiLN15VTHvXe+pULJsl6UidCoEDOWv2pGuKjQucl2OIlK2IH3SIICS9v2fuZFDbTUGfXU8ggFxdG22ZeluxJ0WN8hXtlDGL0rjyXnaKzplFqYyhdV9tOBb0dyytVYhzClHtibe6YlvyYq0CV3NZFfp1qyMpss4uuO3iPl2A1lw7z8KStB+xrUIxmfyRPh1OqnQodSekTXnLfk94PSfriB9K7qJ2NswmC7p6UGU0e2E3CJV51LGn2Htb0A/pNB3ntPM6MnjfuhAp7hyFxvkIOUZQHy7I4izyHKKKicQkBeeJlTeaPbk+dn0s6K9gqLJ7apL0PKvgrDDckaKad8eN59jKkipjMVnQp4YTT61qmYJxkRpnqkA37//2mZxlcpApBybMCEvyvV2Vrm2VreY4qFrV2vCcuNIxP8rl8vOG4wo36MM8i2ElrE8j0N+VZb1p1s5jBitfNqugW22Ul+HRxNCvN151TUHvhTddD5mGho2hA0mKcDDk24BV4IqLDWTAD3HQ/8NtwYM3ftO3bGMQs8IQNFZEeXJdiuqew3BScbx8HsCjHNuZPpATvVAGWunjStRHbMFW6c9U28gb9mFWHJn31VZzcqZahei13czJ8i4/FyMeP1pSOog6FiPej6wgsV8WQYPVcPniCYHkra2ruHhKNnn8S89+XzyW02h5oKkio0OcDcLTdss2ZtkMVCQlE2qT+ic7uef95VYZ8V9BV0xKTAvCu65JUed5q10fYC2ds4FppwKVfYRPgZVB5tHdsyWjW9hf5ci6CD/IefhgM0tiOAtRE6jM0ZJ123KlVWSzTwztjI5klN+wSDCQttFRaJUxsE7LyZX8kFyL4WrDfnILfaDkSOAXaKehqpZp7VV5j4na8lycLz+otjO8pP/uGhX48urt9SehJVy+/LcXr79Lx16+zLKNXF1a9z8F/eh6G5OndjubVlyM9UEHqj0njdzoLYek8WT7ds2wxD7AUZFau34g9uWrtJEs2jWcJNTb0Hsu6ErmfJemzEmZ4AYvR98uRONvP735dEXvbz5dwQ47o2CNYPEUyhsour5s4LPL3ntoQ7PvoB9BByj/Q6QQuZtdHPyfiCY5wWS8lXddxxXV3rXp+mIMDzHFAYll1kW4Uto1KlLo/RbXFzn9hIDje8NhsqDb0ouhJNI2Dq70RWw7iXTJx0XOwhs7iamihj1XrqTQlw2psKRSRfoPWf3Cc+dd0e2pdsa4HVe03ouuhdLrLgqZytLVzTtqlb9nPyvobVJWiGRk9tCPMCZyOR5VZqf2Ib0JOaQe43xih7L72CBKGK7FNR4n0JpLB1PsoKyuHkX9Aja3d71oBZWeVeQKfnhEy+lmc6rYMLgeH/Mrh+CswPBtk0V2Ea2rdL2HF0+npPRBHFyNx6Lbz6nby6O2Gs+dd//kMhbRtQbhJPZdUdYb7BHdA2YthduSykh05Rjwpi3onX6QMaPXXiHKw+1NFgP7tQ1R4rkq79Ume6nsRyJs2cGHwLUiVNvI3nK8SLJVtlq7B2jdkORIKIDJJAlY5oorTFAGcWU/HMdVQVdDlrFEhnWXbkyLL3fkbBJMhA+KifW10ga2hO0tlc4YLjPf7352VW/4vYvvXG+rK++dX9J7B073Zoh9Z9qCxFWwqguNi2d3YN/dWaV93K/4S69MOLtDHqEi6bZD3sLYCuT31rMRjYgu0VbrBxjFvYUn2qguxWAdBq6cBZHrQjLIiju2yN80Iy7eMtMf/iorBx5uVBf+QGs2bic8YlVBPzd6Cz72sM/k7lW4P6qS79PbXeMQMcG/0OvIBf3FqworPfvehsEvY4K2JPIUriZn0ii74eynJKTJfguZMcgFQfhFdC/wZ/VQdPs7SRURBSkwhKGdnc0pONqyh4JLBIIzQlay8cr2Rnkd98uUp4DulExB1RQhbYawsKHOGW7nGXzOJIiwo24Zzvvfzwt6M6Tbw6pjIE0phBwftTFZfdyWPZQ9lM7DM4sSt0wK/hMiBWs/eBehWxjNfpZ22lZud5o3JQ9NbR8irZnW0D3o9QfdcU4r5Azv+k1DDSS6sPT6HGqX/pX0+vz8/II6rJArirCGJVFpgyXfnw/ZxTE3yzLKYRqvloNzlPrk19KzEiwAx4Yk/eWrc4nMiDS4QWLKvl07I+4QJizppI16ETptlzmHSWl7SlkIxUnUpTKSmiM5zb7VqBDp2xQ+xcWqIdMcLHqn4UoHHWyYVNd5Bw2dVrquWUKm5GJzOr5ImnP833JsXDWDV9FRfBy9TPFy47mjlpUVGbepwhsdN2w9pE5weloyaKk/kpo+UKutbpVJqQVb6NESbsKwZHiSa8HZHCuaihH81qhNsoZ3XGqFrCukOIfcKudCBf3SVQgGQa8N5HPYWkkhNtoXIxp2OkqlsPvelkOcVfJnq7xWFrnpzyk5zhZ3BvXY48z3cO61KqOUKdZRxet+Q53XFhHJOhFWKiIkZTq6oMkf6bEHo+mWbSXldeIadCCwqedZaWoGIbh6Djs6ziafdg6u01cLhOX9yG8egxJiqWsZ1h+o1cnqRb5jp5uCQ2yS+ZwgAmdBcsZUqmf3ngIklBLmnEt9KH1gPq0SUxr5VKC5Q8p1jEggy4ljlkI1ea75IdI3fCA+3+3AIjghhWKIousWhreMGt0zhLPPT2b/KOeBux50JpE7wiUOG+XDBjFkLMLu6U4Y8uLueNQs+dlxtSultUSSIS+hTwk/Ec0uG1Y4T4ATxHqYdmzq3ojVqNj7gZXwYGZIRc4Cdf3a6JLefLg+GEns18lxj9yZZLLXNd39dlA/SRfvvhp+sVppq+NqhfiV7jmUxJO7u7tJaeA7b/Ps5YSIaLVCRR1WK7qk6dlKCraz+UzGKq5p2HKa9FzGL987y7O0Hj8MFWkpXaYpo/Vy6mrDUUduV6tpiUoDz6MdPEtRVZowWshfDsdKwfl4/snBl2nWaAPLv7UBhJI2SaelaaMtkPnnTR6vxuB0RMN4YaNC86sLMXi6cIJVgxynj7ic192eTJiJUJ9WnNO071RrxmPPU5nVWyz5QQVejq7o/G+xd3XjTyaMGfu81TKEmsDUz+auW0s1cTprMlzlxk+P18mLn1B0Naf11yqu6JLU6as1XdL69+usOqgr3Omw02MNfqb6qRn9ic7o/+iM/jR6vT7c+jq81WUcXRzJRRl//frfKL8Jc/rmm/sdnkaHY13xG7OfxxBdpxItSLS0JQ/SHpH21ZXfKROSbxm9xPRsueCh/D2R+XX4oHzUyvwrMmE1RNzLrPjTdPJogTIm7VpsOE7v54c1wq+tJGv3c9qieBLFgXcO09mIgTeWb+pnGc434pvGzJIt09shPPw+N6/sPtF/JPdIaz7uuUZzNtzl1FTyLjCj2dlw7Z9U+IgZz7k2P3RcRq7k9K8vfzJIl6eTfxcrhOKjrj/e/dl8GC53yofTa2R2IBDkTKqP2gSJFQMggByUatVq5JeSeIZHiUkxBK476m3pbKUTzA5MLrghLc3J9Ul8uhNpq69epvQZFVbvOcOWw+qvjr47lsE7JbDUUCfQDpXzsX5GbZfT3ILeHDoQg3nl+CoJoKR5nefFIRdlu9XeWSmWpI5JCd9Jlh13bjkUYhlIV4c8ORVMskogN51BohH8ijS+dVvk7lKgoJJ3pTM0FZw5YVtSr5aqCxfD3JTAo5LtQL3AXVzNJi8L+tvNj9h9eqiWz1MpK4Vm9CgKMsIsCH8uSHKltqS6t4KLAN2DwaAW7JRXLUeBow+NNor7DoMCOC1aDkGu7NUmdzFuBU94jDP2wvqNjqnOXWToVgoNECw1Z61LiuxbwT+GbqqAq/MBtkjLjiAFMvzUchuaQUBDlK20VKkyW1HUmyYB++RRUAvuhA6JcybQ9Jm9sGG3jDqAlU+0x2ZSBA79p6Xgxijgjj3anUtMQEWPArkWPMM6wSKlzhuaZBUqDvonkBtguBsVwYqEXeebGiO6meswt7MJ80R5LHgnUHJMSL1U5o59MXlV0MerDx9v3v7y40hpvj/ozN+lc3CKmG+1gsGMZDoA7VIFpVpcen4hMHyHkMGNQm/IP403pIKtt09B00sqq3H9KDhbUokTwuYCS1WP0CjoxnWNnUtlYalrKbO9q/oy61FuR5aqm1PnXckJN5U2j7J7uBmhSux3Ddzr2BIpJt8WaebR6HKHBsjpoWt8XAFPAEkMvWCajgSlLf0DqL+Cuny+SMAPTOa/gE9wAABR0C8o8Q/NytS5OnQoqXSdPjZPj0aeiuNXuNpr6q3+0gNltxBAansLlC7+Dx7zg+eakcsKCJn6aaNTUxNKLFQUT4A9Z1m4fZyGaJK0QMUxkWgNQhpJ6RNqLe0VoWvU004rxVxJbdCsApkJ9xoMEdQjxWrRwxhuoLdQlrJx/tjFlw7BAUWVwYLe1JHRzS8bofv0bM7YW5VJcJCYBBZba98Wk9cF/efVx+t3n5/0uh8FhcjQjneDm+rtqYofUeJBtxftACMvvgxg+gg2B3XTMEvnfY31Z7Rw+hRQvZSmi0DtCbtlqnsETEFpBVdCC1S6tHISDAAWDtXUcUBBM642+zWrAwYs6GmiFovQfyqBdh8UMiPSu4YlMI9pFwOoSMcLweWSecImkvB9bwe5vZLd0W3UD1TuS4kSITp8ilHljuHtLz/8fP3pGGrEhrfsA0IZMpVGbbmYfFfkmQX9CGUffzUhLur004mTRrTQIn0rIMjSUhAjrHQs6PbQVjtp+ZbKZm8k3buLFBoOm8hpQWgs0UlGPxd4KUHah56qmGCNQLBzvanI4MsftINzK1q+6pGsLH9/g7Zt6pxncxLS696LEHTbebcVJDcMLE7OQ76iOGGA584cjDDuXOovh75t0biT7CZ/3zDsfujip1bvO+fhg48XnXzkjfKV4YCYRZIGzVO+M2qbh8btAr6nEdqUpT+PNiYH2PYw9P05BUaGikSpVRow5xxzZNNB9ojOVcaM/3yOj4VK+SwiheDRl05ZtfihMw5+bAnG6JDUCsKWLSXy27nYuQTdhsv7JD3Za8zG1BaSalZUOPd6xJXJDaxLHd8LIOXjz6Yy4JgX/C97V0z+H1BLAQIUABQAAAAIAAAAIVwJEt0MXAIAAAgEAAAKAAAAAAAAAAAAAACkAQAAAABhZ2VudC55YW1sUEsBAhQAFAAAAAgAAAAhXN159oETAgAAkAMAABUAAAAAAAAAAAAAAKQBhAIAAGNvbmZpZ3Mvc2FtcGxpbmcueWFtbFBLAQIUABQAAAAIAAAAIVz68iwMRgIAACoEAAAQAAAAAAAAAAAAAACkAcoEAABldmFsX2NvbmZpZy55YW1sUEsBAhQAFAAAAAgAAAAhXJbmrUNuDwAAlyYAABQAAAAAAAAAAAAAAKQBPgcAAHByb21wdHMvc3dlX2NvZGVyLm1kUEsFBgAAAAAEAAQA+wAAAN4WAAAAAA=="
zipfile.ZipFile(io.BytesIO(base64.b64decode(SUBMISSION_B64))).extractall(AGENT_DIR)

# Load tasks from the published dataset
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)

print(f'Data directory: {DATA_DIR}')
print(f'Agent directory: {AGENT_DIR} (model={MODEL_NAME})')
print(f'Loaded {len(tasks)} tasks from {TASKS_PATH.name}')
for t in tasks[:5]:
    print(f'  - {t.instance_id} ({t.repo} @ {t.base_commit[:8]})')

## 3. Code Graph Functions

In [ ]:
from swegemma import graph as sg

sample_task = tasks[0]
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')

# Load the pre-computed repository code graph and node embeddings for the sample task
repo_graph = sg.get_graph(
    repo_name=sample_task.repo,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
sample_nodes = list(repo_graph.nodes())
print(f'Graph for {sample_task.repo} ({sample_task.instance_id}):')
print(f'  Nodes: {repo_graph.number_of_nodes()}, Edges: {repo_graph.number_of_edges()}')

# Select a connected symbol and query structural neighbors (callers, callees, definitions)
query_node = next((n for n, deg in repo_graph.degree() if deg >= 5), sample_nodes[0])
neighbors = sg.get_neighbor(
    node=query_node,
    graph=repo_graph,
    max_neighbors=10,
)
print(f'\nNeighbors of {query_node!r} (showing up to 5):')
for n in neighbors[:5]:
    print(f'  - {n}')

# Search for semantically similar code nodes using pre-computed vector embeddings
similar = sg.get_similar_nodes(
    node=query_node,
    repo_name=sample_task.repo,
    k=5,
    graph=repo_graph,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
print(f'\nTop similar nodes to {query_node!r}:')
for item in similar:
    print(f"  - {item['node_name']} (similarity={item['similarity']:.4f})")

# Extract an induced subgraph connecting the query symbol and its neighbors
focal_nodes = [query_node, *neighbors[:4]]
subgraph = sg.get_induced_subgraph(repo_graph, focal_nodes)
print(
    f'\nInduced subgraph over {len(focal_nodes)} focal nodes: '
    f'{subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges'
)

## 4. Start vLLM Server

In [ ]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

litellm.drop_params = True

ALLOWED_MODEL_NAMES = frozenset({
    'gemma-4-31b-it-qat-w4a16-ct',
    'gemma-4-12b-it-qat-w4a16-ct',
    'gemma-4-e4b-it-qat-w4a16-ct',
    'gemma-4-e2b-it-qat-w4a16-ct',
})
MODEL_VLLM_EXTRA_ARGS = {
    'gemma-4-12b-it-qat-w4a16-ct': (
        '--hf-overrides',
        '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}',
    ),
}
_SAFE_PATH_COMPONENT = re.compile(r'[A-Za-z0-9][A-Za-z0-9._-]*')
INFERENCE_API_KEY = os.environ.get('LOCAL_API_KEY', 'EMPTY')

# Validate single canonical base model and discover any PEFT LoRA adapters in the submission directory.
# Note: sample_submission includes dummy LoRA adapters (adapters/main_lora and adapters/tool_lora)
# shaped for gemma-4-31b-it-qat-w4a16-ct; Cell 2 removes them automatically when MODEL_NAME != 31B.
declared_models = discover_declared_models(AGENT_DIR)
assert len(declared_models) == 1, f'Expected exactly 1 declared model, got: {sorted(declared_models)}'
declared_model = next(iter(declared_models))
assert declared_model in ALLOWED_MODEL_NAMES, f'Unsupported model {declared_model!r}. Allowed: {sorted(ALLOWED_MODEL_NAMES)}'

MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{declared_model}/2')
assert MODEL_PATH.exists(), f'Model weights not found at {MODEL_PATH}. Ensure the model input is attached.'

adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
adapter_map = getattr(adapters, 'adapters', None) or {}
for adapter_name in adapter_map:
    assert _SAFE_PATH_COMPONENT.fullmatch(adapter_name), (
        f'Unsafe adapter name {adapter_name!r}: must match [A-Za-z0-9][A-Za-z0-9._-]*'
    )
detected_ranks = [
    r for info in adapter_map.values() if (r := getattr(info, 'rank', None)) is not None and r > 0
]

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

if 'server_instance' in globals() and server_instance is not None:
    server_instance.stop()

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=bool(adapter_map),
    max_loras=max(1, len(adapter_map)),
    max_lora_rank=max(detected_ranks) if detected_ranks else 128,
    scheduler_reserve_full_isl=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
    extra_args=MODEL_VLLM_EXTRA_ARGS.get(declared_model, ()),
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

# Register model and LoRA adapter aliases for Google ADK
models = server_instance.create_model_registry(
    aliases=[declared_model],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

## 5. Run Phase 1 Inference and Phase 2 Verification

In [ ]:
import asyncio
import concurrent.futures
import pandas as pd
import yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator


def run_sync(coro_or_fn, *args, **kwargs):
    """Run an async coroutine synchronously inside a notebook event loop."""
    if callable(coro_or_fn) and not asyncio.iscoroutine(coro_or_fn):
        fn = lambda: coro_or_fn(*args, **kwargs)
    else:
        fn = lambda: coro_or_fn
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())


# Read optional evaluation settings from the submission's eval_config.yaml / eval_config.yml
eval_config_file = next(
    (c for name in ('eval_config.yaml', 'eval_config.yml') if (c := AGENT_DIR / name).is_file()),
    None,
)
raw_eval_cfg = (
    yaml.safe_load(eval_config_file.read_text(encoding='utf-8')) or {}
    if eval_config_file is not None
    else {}
)
eval_section = raw_eval_cfg.get('evaluation', raw_eval_cfg)

timeout_seconds = int(eval_section.get('timeout_seconds', 300))
verification_timeout_seconds = int(eval_section.get('verification_timeout_seconds', 300))
max_tool_calls = int(eval_section.get('max_tool_calls', 100))
max_time_minutes = float(eval_section.get('max_time_minutes', 60.0))
turns_raw = eval_section.get('max_turns', eval_section.get('max_llm_calls'))
max_turns = int(turns_raw) if turns_raw is not None else None

# EXP-003 shim check (easy cohort): exp002 picked up 3 known inline_snapshot/dirty_equals
# tasks but all 3 hit the 4.5 min wall-clock cap in LOCATE/REPRODUCE before ever running pytest
# (experiments/README.md 1k) - the shim was never exercised. These 3 have the smallest reference
# patches (single-file, <700 chars) of the 25 known-affected tasks, to reach pytest collection
# fast and actually trigger the ModuleNotFoundError the shim is meant to recover from.
TARGET_IDS = {'fastapi_14361', 'fastapi_14360', 'fastapi_14303'}
SAMPLE_TASKS = [t for t in tasks if t.instance_id in TARGET_IDS]
assert len(SAMPLE_TASKS) == len(TARGET_IDS), f'missing tasks: {TARGET_IDS - {t.instance_id for t in SAMPLE_TASKS}}'
limits, gen_constraints = build_submission_limits()

# Configure Evaluator to run Phase 1 (agent patch generation) and Phase 2 (verification)
# using the subprocess sandbox backend and the submission's eval_config parameters
eval_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=WORKING_DIR / 'results',
    submission_dir=AGENT_DIR,
    models=models,
    sandbox='subprocess',
    timeout_seconds=timeout_seconds,
    verification_timeout_seconds=verification_timeout_seconds,
    max_time_minutes=max_time_minutes,
    max_tool_calls=max_tool_calls,
    max_turns=max_turns,
    limits=limits,
    generation_constraints=gen_constraints,
    adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15,
        overlap_size=2,
        token_threshold=14336,
        event_retention_size=5,
    ),
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    wheels_dir=DATA_DIR / 'wheels',
    verbose=False,
)

evaluator = Evaluator(eval_config)
predictions = []

for idx, task in enumerate(SAMPLE_TASKS, start=1):
    print(f'[{idx}/{len(SAMPLE_TASKS)}] Evaluating {task.instance_id} ({task.repo})...')
    result = run_sync(
        evaluator.evaluate_task,
        task=task,
        task_index=idx,
        total_tasks=len(SAMPLE_TASKS),
    )
    patch = (result.agent_patch or '').strip()
    predictions.append({'id': task.instance_id, 'prediction': patch if patch else 'NO_PATCH'})
    print(
        f'  -> resolved={result.resolved}, '
        f'exit_code={result.test_exit_code}, '
        f'patch_chars={len(patch)}, '
        f'tool_calls={result.tool_calls}, '
        f'duration={result.duration_seconds:.1f}s'
    )

submission_df = pd.DataFrame(predictions, columns=['id', 'prediction'])
display(submission_df)

## 6. Package Submission Archive

In [ ]:
import zipfile
from adk_submission import validate_directory
from swegemma.config import ALLOWED_SUBMISSION_EXTENSIONS, MAX_SUBMISSION_SIZE_BYTES

# Validate unpacked submission directory structure, file types, symlinks, and size limits
for dirpath, dirnames, filenames in os.walk(AGENT_DIR, followlinks=False):
    parent = Path(dirpath)
    for name in (*dirnames, *filenames):
        assert not (parent / name).is_symlink(), f'Symlinks are not permitted: {(parent / name).relative_to(AGENT_DIR)}'
validate_directory(AGENT_DIR, limits=limits)

# Create submission.zip with agent.yaml at the root of the archive
zip_base = WORKING_DIR / 'submission'
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=AGENT_DIR))

# Validate archive contents against competition constraints
with zipfile.ZipFile(zip_path, 'r') as zf:
    infos = zf.infolist()
    total_size = sum(i.file_size for i in infos)
    assert total_size <= MAX_SUBMISSION_SIZE_BYTES, f'Archive exceeds 3 GiB limit: {total_size}'
    for info in infos:
        if not info.is_dir():
            ext = Path(info.filename).suffix.lower()
            assert ext in ALLOWED_SUBMISSION_EXTENSIONS, f'Disallowed file extension: {info.filename}'

print(f'Created {zip_path} ({zip_path.stat().st_size / (1024 * 1024):.2f} MiB)')